## This is the actual start for the code


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    auc,
)

from src.data_utils import load_training_dataframe
from src.models.bert import BertTrainingConfig, train_bert


In [ ]:
df = load_training_dataframe(
    data_path="./data/train.csv",
    sample_size=20000,
    min_text_length=50,
)

print("Prepared shape:", df.shape)
df.head()


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 1),
    min_df=5,
)

X = vectorizer.fit_transform(df["full_text"])
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Feature Matrix Shape:", X.shape)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5),
    "Linear SVM": LinearSVC(),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        n_estimators=200,
    ),
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
    ])

    print(f"{name} done")


In [ ]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"],
)
results_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.xticks(rotation=45)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["F1 Score"])
plt.xticks(rotation=45)
plt.title("Model F1 Score Comparison")
plt.ylabel("F1 Score")
plt.show()


In [ ]:
best_model_name = results_df.sort_values("F1 Score", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)

print("Best Model:", best_model_name)

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()


In [ ]:
param_grid = {"C": [0.1, 1, 10]}

grid = GridSearchCV(LinearSVC(), param_grid, cv=3, scoring="f1")
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Cross-Validation F1:", grid.best_score_)


In [ ]:
print(classification_report(y_test, y_pred_best))


In [ ]:
logistic_model = models["Logistic Regression"]
y_prob = logistic_model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.show()


In [ ]:
importances = models["XGBoost"].feature_importances_
indices = np.argsort(importances)[-20:]

plt.figure(figsize=(8, 6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), [vectorizer.get_feature_names_out()[i] for i in indices])
plt.title("Top 20 Important Words (XGBoost)")
plt.show()


In [ ]:
errors = y_test != y_pred_best
misclassified_indices = y_test[errors].index

print("Number of misclassified samples:", int(errors.sum()))
df.loc[misclassified_indices][["full_text", "label"]].head()


In [ ]:
feature_names = vectorizer.get_feature_names_out()
log_model = models["Logistic Regression"]
coefs = log_model.coef_[0]

top_fake_indices = np.argsort(coefs)[-20:]
top_real_indices = np.argsort(coefs)[:20]

print("Top 20 Words Predicting FAKE News:
")
print([feature_names[i] for i in top_fake_indices])

print("
Top 20 Words Predicting REAL News:
")
print([feature_names[i] for i in top_real_indices])


In [ ]:
def predict_news(text):
    text_vector = vectorizer.transform([text])
    model = models["Logistic Regression"]

    fake_prob = model.predict_proba(text_vector)[0][1]
    real_prob = 1 - fake_prob
    threshold = 0.85

    if fake_prob >= threshold:
        print("Prediction: FAKE News")
    else:
        print("Prediction: REAL News")

    print(f"Fake Probability: {fake_prob:.4f}")
    print(f"Real Probability: {real_prob:.4f}")


In [ ]:
predict_news("Breaking shocking secret government exposed click here to know truth")


In [ ]:
predict_news("The government released an official statement regarding economic policy changes")


In [ ]:
fake_text = df[df["label"] == 1]["full_text"]
real_text = df[df["label"] == 0]["full_text"]

fake_words = Counter(" ".join(fake_text).split())
real_words = Counter(" ".join(real_text).split())

print("Most common words in FAKE news:")
print(fake_words.most_common(10))

print("
Most common words in REAL news:")
print(real_words.most_common(10))


## BERT experiment

This notebook now uses the repo BERT training module instead of notebook-only `!pip install` cells. The first run may download `bert-base-uncased`.


In [ ]:
bert_config = BertTrainingConfig(
    model_name="bert-base-uncased",
    data_path="./data/train.csv",
    sample_size=4000,
    epochs=1,
    output_dir="artifacts/bert_notebook",
)

bert_report = train_bert(bert_config)
bert_report["final_metrics"]


## Save the classical model used by the app


In [ ]:
deployment_model = models["Logistic Regression"]

joblib.dump(vectorizer, "vectorizer.pkl")
joblib.dump(deployment_model, "fake_news_model.pkl")

print("Saved Logistic Regression model and vectorizer for the app.")
